# 🗄️ Notebook 2 — Data Warehouse & Star Schema Modeling
**Project:** Shopify E-Commerce Analytics Data Warehouse  
**Layer:** Warehouse (Dimensional Model)  
**Purpose:** Build the star schema in-memory using pandas, define surrogate keys, map source→dimensions, validate relationships.

---

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('shopify_cleaned.csv', parse_dates=['order_date'])
print(f"✅ Loaded cleaned dataset: {df.shape[0]:,} rows × {df.shape[1]} cols")

✅ Loaded cleaned dataset: 60,000 rows × 21 cols


## 1. Warehouse Design Decisions

In [2]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║           STAR SCHEMA DESIGN RATIONALE                               ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  SCHEMA TYPE : Star Schema (not snowflake)                           ║
║  REASON      : Optimized for BI tool query performance.              ║
║                One join from fact to any dimension.                  ║
║                Readable SQL. Denormalization is intentional.         ║
║                                                                      ║
║  FACT TABLE  : fact_sales                                            ║
║  GRAIN       : One row per order_id (single product per order)       ║
║                Confirmed by duplicate check in Notebook 1            ║
║                                                                      ║
║  DIMENSIONS  : dim_date        — Calendar/fiscal attributes          ║
║                dim_customer    — Customer profile + lifecycle        ║
║                dim_product     — Product + category attributes       ║
║                dim_country     — Geography + region mapping          ║
║                dim_traffic_source — Channel classification           ║
║                dim_payment_method — Payment instrument type          ║
║                                                                      ║
║  SURROGATE KEYS: Integer sequences (1,2,3…) in all dim tables        ║
║  NATURAL KEYS  : Preserved as business keys for traceability         ║
║  SCD TYPE      : Type 1 (overwrite) for all dimensions               ║
╚══════════════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════════════╗
║           STAR SCHEMA DESIGN RATIONALE                               ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  SCHEMA TYPE : Star Schema (not snowflake)                           ║
║  REASON      : Optimized for BI tool query performance.              ║
║                One join from fact to any dimension.                  ║
║                Readable SQL. Denormalization is intentional.         ║
║                                                                      ║
║  FACT TABLE  : fact_sales                                            ║
║  GRAIN       : One row per order_id (single product per order)       ║
║                Confirmed by duplicate check in Notebook 1            ║
║                                                                      ║
║  DIMENSIONS  : dim_date        — Calendar/fiscal

## 2. Build dim_date

In [3]:
# ── Date spine covering full dataset range ─────────────────────────
date_range = pd.date_range(start='2023-01-01', end='2025-12-31', freq='D')
dim_date = pd.DataFrame({'full_date': date_range})
dim_date['date_key']      = dim_date['full_date'].dt.strftime('%Y%m%d').astype(int)
dim_date['year']          = dim_date['full_date'].dt.year
dim_date['quarter']       = dim_date['full_date'].dt.quarter
dim_date['quarter_label'] = 'Q' + dim_date['quarter'].astype(str) + ' ' + dim_date['year'].astype(str)
dim_date['month']         = dim_date['full_date'].dt.month
dim_date['month_name']    = dim_date['full_date'].dt.strftime('%B')
dim_date['month_label']   = dim_date['full_date'].dt.strftime('%b %Y')
dim_date['week_of_year']  = dim_date['full_date'].dt.isocalendar().week.astype(int)
dim_date['day_of_month']  = dim_date['full_date'].dt.day
dim_date['day_of_week']   = dim_date['full_date'].dt.dayofweek
dim_date['day_name']      = dim_date['full_date'].dt.strftime('%A')
dim_date['is_weekend']    = dim_date['day_of_week'].isin([5,6])
dim_date['is_weekday']    = ~dim_date['is_weekend']

print(f"✅ dim_date built: {dim_date.shape[0]:,} rows")
print(f"   Date range: {dim_date['full_date'].min().date()} → {dim_date['full_date'].max().date()}")
dim_date.head(3)

✅ dim_date built: 1,096 rows
   Date range: 2023-01-01 → 2025-12-31


,full_date,date_key,year,quarter,quarter_label,month,month_name,month_label,week_of_year,day_of_month,day_of_week,day_name,is_weekend,is_weekday
0,2023-01-01,20230101,2023,1,Q1 2023,1,January,Jan 2023,52,1,6,Sunday,True,False
1,2023-01-02,20230102,2023,1,Q1 2023,1,January,Jan 2023,1,2,0,Monday,False,True
2,2023-01-03,20230103,2023,1,Q1 2023,1,January,Jan 2023,1,3,1,Tuesday,False,True


## 3. Build dim_customer

In [4]:
# ── Aggregate customer profile from transactions ───────────────────
cust_agg = df.groupby('customer_id').agg(
    customer_country = ('customer_country', lambda x: x.iloc[-1]),
    first_order_date = ('order_date', 'min'),
    last_order_date  = ('order_date', 'max'),
    total_orders     = ('order_id', 'nunique'),
    total_revenue    = ('revenue', 'sum'),
    total_profit     = ('profit', 'sum'),
    avg_rating       = ('rating', 'mean'),
).reset_index()

cust_agg['avg_order_value']     = (cust_agg['total_revenue'] / cust_agg['total_orders']).round(2)
cust_agg['is_repeat_customer']  = cust_agg['total_orders'] > 1
cust_agg['customer_lifespan_days'] = (cust_agg['last_order_date'] - cust_agg['first_order_date']).dt.days
cust_agg['total_revenue']       = cust_agg['total_revenue'].round(2)
cust_agg['total_profit']        = cust_agg['total_profit'].round(2)

# Assign surrogate key
cust_agg = cust_agg.sort_values('customer_id').reset_index(drop=True)
cust_agg.insert(0, 'customer_key', range(1, len(cust_agg)+1))

dim_customer = cust_agg.copy()
print(f"✅ dim_customer: {dim_customer.shape[0]:,} rows")
print(f"   Repeat customers : {dim_customer['is_repeat_customer'].sum():,} ({dim_customer['is_repeat_customer'].mean()*100:.1f}%)")
dim_customer.head(3)

✅ dim_customer: 31,154 rows
   Repeat customers : 17,832 (57.2%)


,customer_key,customer_id,customer_country,first_order_date,last_order_date,total_orders,total_revenue,total_profit,avg_rating,avg_order_value,is_repeat_customer,customer_lifespan_days
0,1,10000,Australia,2024-02-10,2024-02-10,1,133.15,119.28,4.20,133.15,False,0
1,2,10001,India,2023-01-01,2025-04-28,5,4950.25,4858.84,2.54,990.05,True,848
2,3,10002,USA,2023-04-02,2024-06-11,2,552.04,523.62,3.50,276.02,True,436


## 4. Build dim_product

In [5]:
# ── Product profile from transactions ─────────────────────────────
prod_agg = df.groupby(['product_id','product_category']).agg(
    avg_list_price   = ('product_price', 'mean'),
    avg_discount_pct = ('discount_percent', 'mean'),
    total_units_sold = ('quantity', 'sum'),
    total_revenue    = ('revenue', 'sum'),
    total_profit     = ('profit', 'sum'),
    return_rate      = ('is_returned', 'mean'),
    avg_rating       = ('rating', 'mean'),
    order_count      = ('order_id', 'nunique'),
).reset_index().round(2)

prod_agg = prod_agg.sort_values('product_id').reset_index(drop=True)
prod_agg.insert(0, 'product_key', range(1, len(prod_agg)+1))
dim_product = prod_agg.copy()

print(f"✅ dim_product: {dim_product.shape[0]:,} rows")
print(f"   Categories: {dim_product['product_category'].nunique()}")
dim_product.head(3)

✅ dim_product: 34,658 rows
   Categories: 7


,product_key,product_id,product_category,avg_list_price,avg_discount_pct,total_units_sold,total_revenue,total_profit,return_rate,avg_rating,order_count
0,1,1000,Accessories,382.76,30.00,2,535.86,503.54,0.50,2.65,2
1,2,1000,Footwear,286.89,13.33,12,3002.89,2969.77,0.33,3.17,3
2,3,1000,Home Decor,770.68,10.00,4,2774.44,2755.93,0.00,1.80,1


## 5. Build dim_country, dim_traffic_source, dim_payment_method

In [6]:
# ── dim_country ────────────────────────────────────────────────────
region_map = {
    'USA':'North America','Canada':'North America','UK':'Europe',
    'UAE':'Middle East','India':'South Asia','Germany':'Europe',
    'France':'Europe','Australia':'Oceania'
}
currency_map = {
    'USA':'USD','Canada':'CAD','UK':'GBP','UAE':'AED',
    'India':'INR','Germany':'EUR','France':'EUR','Australia':'AUD'
}
countries = pd.DataFrame({'country_name': df['customer_country'].unique()})
countries['region']        = countries['country_name'].map(region_map).fillna('Other')
countries['currency_code'] = countries['country_name'].map(currency_map).fillna('USD')
countries = countries.sort_values('country_name').reset_index(drop=True)
countries.insert(0, 'country_key', range(1, len(countries)+1))
dim_country = countries.copy()

# ── dim_traffic_source ─────────────────────────────────────────────
channel_type_map = {
    'Paid Ads':'Paid','Email':'Owned','Social Media':'Paid/Organic',
    'Organic':'Organic','Direct':'Direct'
}
paid_map = {'Paid Ads':True,'Social Media':True,'Email':False,'Organic':False,'Direct':False}
sources = pd.DataFrame({'traffic_source': df['traffic_source'].unique()})
sources['channel_type'] = sources['traffic_source'].map(channel_type_map).fillna('Other')
sources['is_paid']      = sources['traffic_source'].map(paid_map).fillna(False)
sources = sources.sort_values('traffic_source').reset_index(drop=True)
sources.insert(0, 'traffic_source_key', range(1, len(sources)+1))
dim_traffic_source = sources.copy()

# ── dim_payment_method ─────────────────────────────────────────────
ptype_map = {
    'Credit Card':'Card','Debit Card':'Card','PayPal':'Digital Wallet',
    'Apple Pay':'Digital Wallet','Cash on Delivery':'COD'
}
digital_map = {'PayPal':True,'Apple Pay':True,'Credit Card':False,
               'Debit Card':False,'Cash on Delivery':False}
payments = pd.DataFrame({'payment_method': df['payment_method'].unique()})
payments['payment_type']      = payments['payment_method'].map(ptype_map).fillna('Other')
payments['is_digital_wallet'] = payments['payment_method'].map(digital_map).fillna(False)
payments = payments.sort_values('payment_method').reset_index(drop=True)
payments.insert(0, 'payment_method_key', range(1, len(payments)+1))
dim_payment_method = payments.copy()

print(f"✅ dim_country        : {dim_country.shape[0]} rows")
print(f"✅ dim_traffic_source : {dim_traffic_source.shape[0]} rows")
print(f"✅ dim_payment_method : {dim_payment_method.shape[0]} rows")

✅ dim_country        : 7 rows
✅ dim_traffic_source : 5 rows
✅ dim_payment_method : 5 rows


## 6. Build fact_sales

In [7]:
# ── Map natural keys → surrogate keys ──────────────────────────────
fact = df.copy()
fact['date_key'] = fact['order_date'].dt.strftime('%Y%m%d').astype(int)

fact = fact.merge(dim_customer[['customer_id','customer_key']],     on='customer_id')
fact = fact.merge(dim_product[['product_id','product_key']],         on='product_id')
fact = fact.merge(dim_country[['country_name','country_key']],
                  left_on='customer_country', right_on='country_name')
fact = fact.merge(dim_traffic_source[['traffic_source','traffic_source_key']], on='traffic_source')
fact = fact.merge(dim_payment_method[['payment_method','payment_method_key']], on='payment_method')

# Derived measures
fact['discount_amount']   = ((fact['product_price'] - fact['discounted_price']) * fact['quantity']).round(2)
fact['gross_revenue']     = (fact['product_price'] * fact['quantity']).round(2)
fact['profit_margin_pct'] = (fact['profit'] / fact['revenue'].replace(0, np.nan)).round(4)

# Select final fact columns
FACT_COLS = [
    'order_id','date_key','customer_key','product_key',
    'country_key','traffic_source_key','payment_method_key',
    'quantity','product_price','discounted_price','discount_percent',
    'discount_amount','shipping_cost','revenue','profit',
    'gross_revenue','profit_margin_pct','rating','is_returned'
]
fact_sales = fact[FACT_COLS].copy()
fact_sales.insert(0, 'sales_key', range(1, len(fact_sales)+1))

print(f"✅ fact_sales built: {fact_sales.shape[0]:,} rows × {fact_sales.shape[1]} cols")
print(f"\nFACT TABLE GRAIN: one row per order_id")
print(f"Total revenue  : ${fact_sales['revenue'].sum():,.2f}")
print(f"Total profit   : ${fact_sales['profit'].sum():,.2f}")
fact_sales.head(3)

✅ fact_sales built: 314,418 rows × 20 cols

FACT TABLE GRAIN: one row per order_id
Total revenue  : $312,228,580.29
Total profit   : $307,975,486.02


,sales_key,order_id,date_key,customer_key,product_key,country_key,traffic_source_key,payment_method_key,quantity,product_price,discounted_price,discount_percent,discount_amount,shipping_cost,revenue,profit,gross_revenue,profit_margin_pct,rating,is_returned
0,1,1,20230413,3876,33809,7,4,3,1,143.39,86.03,40.0,57.36,21.08,86.03,64.95,143.39,0.755,2.0,0
1,2,1,20230413,3876,33810,7,4,3,1,143.39,86.03,40.0,57.36,21.08,86.03,64.95,143.39,0.755,2.0,0
2,3,1,20230413,3876,33811,7,4,3,1,143.39,86.03,40.0,57.36,21.08,86.03,64.95,143.39,0.755,2.0,0


## 7. Referential Integrity Validation

In [8]:
# ── Validate all FK joins ──────────────────────────────────────────
checks = [
    ('date_key',           dim_date,           'date_key'),
    ('customer_key',       dim_customer,        'customer_key'),
    ('product_key',        dim_product,         'product_key'),
    ('country_key',        dim_country,         'country_key'),
    ('traffic_source_key', dim_traffic_source,  'traffic_source_key'),
    ('payment_method_key', dim_payment_method,  'payment_method_key'),
]
print("REFERENTIAL INTEGRITY CHECKS")
print("=" * 50)
all_pass = True
for fk, dim, pk in checks:
    orphans = ~fact_sales[fk].isin(dim[pk])
    status  = "✅ PASS" if orphans.sum() == 0 else f"❌ FAIL ({orphans.sum()} orphans)"
    print(f"  fact_sales.{fk:<24} → {status}")
    if orphans.sum(): all_pass = False

print("=" * 50)
print(f"  Overall: {'✅ ALL CHECKS PASSED' if all_pass else '❌ ISSUES FOUND'}")

REFERENTIAL INTEGRITY CHECKS
  fact_sales.date_key                 → ✅ PASS
  fact_sales.customer_key             → ✅ PASS
  fact_sales.product_key              → ✅ PASS
  fact_sales.country_key              → ✅ PASS
  fact_sales.traffic_source_key       → ✅ PASS
  fact_sales.payment_method_key       → ✅ PASS
  Overall: ✅ ALL CHECKS PASSED


## 8. Star Schema Diagram (ASCII)

In [9]:
print("""
                          ┌─────────────────┐
                          │   dim_date      │
                          │  (date_key PK)  │
                          └────────┬────────┘
                                   │ FK
    ┌──────────────┐   FK ┌────────┴─────────┐ FK  ┌──────────────────┐
    │ dim_customer │──────│   fact_sales     │─────│   dim_product    │
    │(customer_key)│      │                  │     │  (product_key)   │
    └──────────────┘      │  sales_key  PK   │     └──────────────────┘
                          │  order_id   DD   │
    ┌──────────────┐  FK  │  date_key   FK   │ FK  ┌──────────────────┐
    │ dim_country  │──────│  customer_key FK │─────│dim_traffic_source│
    │(country_key) │      │  product_key FK  │     │(traffic_src_key) │
    └──────────────┘      │  country_key FK  │     └──────────────────┘
                          │  traffic_src FK  │
                          │  payment_mth FK  │ FK  ┌──────────────────┐
                          │                  │─────│dim_payment_method│
                          │  MEASURES:       │     │(pay_method_key)  │
                          │  quantity        │     └──────────────────┘
                          │  revenue         │
                          │  profit          │
                          │  discount_amount │
                          │  shipping_cost   │
                          │  gross_revenue   │
                          └──────────────────┘

  DD = Degenerate Dimension (no dim table needed)
  PK = Primary Key   FK = Foreign Key
""")


                          ┌─────────────────┐
                          │   dim_date      │
                          │  (date_key PK)  │
                          └────────┬────────┘
                                   │ FK
    ┌──────────────┐   FK ┌────────┴─────────┐ FK  ┌──────────────────┐
    │ dim_customer │──────│   fact_sales     │─────│   dim_product    │
    │(customer_key)│      │                  │     │  (product_key)   │
    └──────────────┘      │  sales_key  PK   │     └──────────────────┘
                          │  order_id   DD   │
    ┌──────────────┐  FK  │  date_key   FK   │ FK  ┌──────────────────┐
    │ dim_country  │──────│  customer_key FK │─────│dim_traffic_source│
    │(country_key) │      │  product_key FK  │     │(traffic_src_key) │
    └──────────────┘      │  country_key FK  │     └──────────────────┘
                          │  traffic_src FK  │
                          │  payment_mth FK  │ FK  ┌──────────────────┐
                          │      

## 9. Save Warehouse Tables

In [10]:
import os
os.makedirs('warehouse', exist_ok=True)

fact_sales.to_csv('warehouse/fact_sales.csv',          index=False)
dim_date.to_csv('warehouse/dim_date.csv',               index=False)
dim_customer.to_csv('warehouse/dim_customer.csv',       index=False)
dim_product.to_csv('warehouse/dim_product.csv',         index=False)
dim_country.to_csv('warehouse/dim_country.csv',         index=False)
dim_traffic_source.to_csv('warehouse/dim_traffic_source.csv', index=False)
dim_payment_method.to_csv('warehouse/dim_payment_method.csv', index=False)

print("✅ All warehouse tables saved to /warehouse/")
for name, tbl in [
    ('fact_sales',fact_sales),('dim_date',dim_date),
    ('dim_customer',dim_customer),('dim_product',dim_product),
    ('dim_country',dim_country),('dim_traffic_source',dim_traffic_source),
    ('dim_payment_method',dim_payment_method)
]:
    print(f"   {name:<25} {tbl.shape[0]:>7,} rows  {tbl.shape[1]:>2} cols")

✅ All warehouse tables saved to /warehouse/
   fact_sales                314,418 rows  20 cols
   dim_date                    1,096 rows  14 cols
   dim_customer               31,154 rows  12 cols
   dim_product                34,658 rows  11 cols
   dim_country                     7 rows   4 cols
   dim_traffic_source              5 rows   4 cols
   dim_payment_method              5 rows   4 cols
